In [1]:
import datetime
import logging  # ログ制御用
import warnings  # 警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Node.js Theme Palette)
# =====================================================================
def show_market_performances(market_dict, title="Market Performance Overview"):
    today = datetime.date.today()
    # 1年分（約370日）のデータを取得して過去騰落率を正確に計算
    start_date = today - datetime.timedelta(days=370)
    
    # Node.js テーマ風のタイトルヘッダー (Dark Background & Neon Green Text)
    title_html = f"""
    <div style='
        background-color: #181818; 
        border-left: 5px solid #68a063; 
        padding: 12px 18px; 
        border-radius: 4px; 
        margin-bottom: 25px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    '>
        <h2 style='
            margin: 0; 
            color: #539e43; 
            font-family: "Consolas", "Courier New", Monaco, monospace; 
            font-size: 22px;
            letter-spacing: 0.5px;
        '>⬢ {title}</h2>
        <span style='
            font-size: 12px; 
            color: #888888; 
            font-family: monospace;
        '>System Status: ONLINE | As of {today.strftime('%Y-%m-%d')}</span>
    </div>
    """
    display(HTML(title_html))
    
    # 数値フォーマット設定
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # Node.jsテーマに合わせたハイライト関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        # プラス（上昇）：Node.jsのネオングリーン背景
        if val >= 10:
            return 'background-color: #1e3a1e; color: #41d888; font-weight: bold; border-radius: 2px;'
        # マイナス（下落）：ダークレッド/コーラル系
        elif val <= -10:
            return 'background-color: #3b1818; color: #ff6b6b; font-weight: bold; border-radius: 2px;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # セクションヘッダー（Node Dark Gray + Green Accent）
        section_html = f"""
        <div style='
            margin-top: 20px; 
            margin-bottom: 8px; 
            color: #215732; 
            font-family: "Consolas", monospace; 
            font-weight: bold;
            font-size: 15px;
            border-bottom: 1px solid #333333;
            padding-bottom: 4px;
        '>
            <span style='color: #68a063;'>❯</span> {market_name}
        </div>
        """
        display(HTML(section_html))
        
        for name, ticker in ticker_dict.items():
            try:
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # テーブル全体のスタイル定義（Dark Node UI Style）
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .set_table_styles([
                             # テーブル全体
                             {'selector': '', 'props': [('border-collapse', 'collapse'), ('font-family', 'Consolas, monospace'), ('font-size', '13px'), ('background-color', '#121212'), ('color', '#e0e0e0')]},
                             # ヘッダー行（Dark Gray / Green Accent）
                             {'selector': 'th', 'props': [('background-color', '#1f1f1f'), ('color', '#68a063'), ('border', '1px solid #333'), ('padding', '8px 12px'), ('text-align', 'center')]},
                             # データセル
                             {'selector': 'td', 'props': [('border', '1px solid #282828'), ('padding', '7px 12px'), ('text-align', 'right')]},
                             # ホバー効果
                             {'selector': 'tr:hover', 'props': [('background-color', '#1a261a')]}
                         ])
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,568.92",+0.36%,-1.96%,-1.99%,-3.20%,+2.77%,+10.46%
NASDAQ Composite,^IXIC,"25,372.01",+0.83%,-3.10%,-4.18%,-8.07%,-1.81%,+6.45%
Apple,AAPL,331.90,-1.90%,-4.88%,-4.38%,-0.52%,+15.65%,+23.82%
NVIDIA,NVDA,206.69,+2.29%,-3.26%,-5.16%,-13.61%,-0.32%,+0.15%
Microsoft,MSFT,520.99,+2.74%,-1.73%,-1.95%,-1.57%,+26.37%,+1.59%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"63,081.86",-0.33%,-1.36%,+1.43%,-7.65%,+4.20%,+22.91%
Toyota Motor,7203.T,"3,051.00",-0.62%,+0.00%,-2.12%,+6.81%,+1.17%,-1.13%
Sony Group,6758.T,"3,966.00",+0.20%,+0.23%,-4.09%,+5.68%,+25.00%,-5.47%
Mitsubishi UFJ Financial,8306.T,"3,383.00",-1.86%,+0.18%,+2.39%,-4.54%,+22.93%,+50.42%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,303.83",-1.36%,-1.74%,-2.07%,-6.36%,+7.64%,+19.34%
DBS Group,D05.SI,70.88,-1.16%,-3.96%,-4.11%,-9.78%,+23.41%,+39.62%
UOB,U11.SI,40.46,-0.49%,-2.51%,-2.20%,-7.46%,+12.62%,+20.68%
Singtel,Z74.SI,4.11,+0.00%,-0.24%,+0.24%,-3.29%,-9.38%,+0.87%
CapitaLand Ascendas REIT,A17U.SI,2.24,+0.90%,+2.75%,+1.82%,+0.90%,-8.67%,-15.29%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,213.00",+0.50%,-8.83%,-7.72%,-17.16%,-10.77%,+22.01%
ニデック (6594),6594.T,"1,762.00",-2.17%,-2.27%,+3.16%,-16.77%,-28.02%,-9.50%
未来工業 (7931),7931.T,"3,605.00",+0.14%,+5.41%,+3.44%,+1.12%,+23.71%,+11.74%
東部ネットワーク (9036),9036.T,"1,088.00",+0.37%,-2.07%,-1.98%,-10.23%,-9.86%,+13.18%
ニトリHD (9843),9843.T,"3,407.00",+0.12%,+3.15%,+0.12%,+13.23%,+51.14%,+38.96%
MRK HD (9980),9980.T,85.00,-2.30%,-2.30%,-2.30%,-4.49%,-15.00%,-2.49%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,95.94,-0.23%,-0.22%,-0.16%,+1.67%,-1.60%,-0.55%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.82,+0.61%,+0.61%,+1.86%,-0.61%,-15.44%,-15.01%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.45,+0.00%,-1.21%,-1.61%,+0.82%,-13.03%,+0.10%
Thomson Medical Group (トムソン医療),A50.SI,0.05,-2.08%,+0.00%,+0.00%,-6.00%,-17.54%,-20.34%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,3.98,-0.50%,+0.76%,-1.00%,+0.00%,+2.37%,+2.77%
First REIT (ファースト・リート),AW9U.SI,0.18,+0.57%,-4.35%,-1.12%,-12.00%,-23.47%,-31.88%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.49,+0.00%,-1.02%,-3.00%,+2.11%,-15.04%,-4.37%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.18,+0.00%,+7.27%,+7.27%,+3.10%,+13.85%,+61.37%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.20,+0.00%,+0.00%,+0.00%,+1.01%,-5.57%,-8.52%
Medtecs International (医療用消耗品),546.SI,0.10,-2.00%,-7.55%,-7.55%,-17.65%,-16.24%,-20.97%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.36,+0.00%,+0.00%,+2.86%,-2.70%,+5.88%,+2.98%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,2.99,+1.36%,-3.86%,-8.00%,-16.94%,-18.52%,-3.98%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,0.87,-0.57%,-1.69%,-3.33%,-7.45%,-24.35%,-5.34%
Riverstone,AP4.SI,0.73,-0.68%,-2.03%,-2.03%,-5.84%,+1.25%,-15.11%
ヤンジジャン・シップビルディング(BS6),BS6.SI,4.69,+1.30%,+1.52%,-1.88%,-7.50%,+14.39%,+43.37%
Sembcorp Ind (U96),U96.SI,5.77,-0.86%,-0.86%,-1.70%,-1.70%,-8.32%,-5.33%


<div style="background-color: #EBCB8B; color: #2E3440; padding: 12px; border-radius: 6px; font-weight: bold;">
</div>

In [5]:
MARKET_TICKERS = {
    "Singapore Stocks (Featured)": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone (リバーストーン・クリーンルーム用手袋)": "AP4.SI",
        "ヤンジジャン・シップビルディング (BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,2.99,+1.36%,-3.86%,-8.00%,-16.94%,-18.52%,-3.98%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,0.87,-0.57%,-1.69%,-3.33%,-7.45%,-24.35%,-5.34%
Riverstone (リバーストーン・クリーンルーム用手袋),AP4.SI,0.73,-0.68%,-2.03%,-2.03%,-5.84%,+1.25%,-15.11%
ヤンジジャン・シップビルディング (BS6),BS6.SI,4.69,+1.30%,+1.52%,-1.88%,-7.50%,+14.39%,+43.37%
Sembcorp Ind (U96),U96.SI,5.77,-0.86%,-0.86%,-1.70%,-1.70%,-8.32%,-5.33%


In [6]:
MARKET_TICKERS = {
    # 1. 安定重視・コア保有向け（好財務＋高配当）
    "Singapore Core & Stability (コア保有向け)": {
        "Sembcorp Industries (U96・大手公益/エネルギー)": "U96.SI",
        "Valuetronics Holdings (BN2・電子機器製造受託/無借金)": "BN2.SI",
    },
    # 2. 高利回り・インカムゲイン重視（配当狙い）
    "Singapore High Yield (高利回り狙い)": {
        "Multi-Chem (AWZ・ITセキュリティ/高配当)": "AWZ.SI",
        "Yangzijiang Financial Holding (YF8・金融)": "YF8.SI",
        "Yangzijiang Shipbuilding (BS6・造船/高収益)": "BS6.SI",
    },
    # 3. 高付加価値・成長＆分配金ハイブリッド
    "Singapore Growth & Hybrid (成長＆分配)": {
        "Riverstone Holdings (AP4・クリーンルーム用手袋)": "AP4.SI",
        "Parkway Life REIT (C2PU・医療REIT/安定分配)": "C2PU.SI",
        "Sanli Environmental Limited": "1E3.SI", 
    },
    
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Sembcorp Industries (U96・大手公益/エネルギー),U96.SI,5.77,-0.86%,-0.86%,-1.70%,-1.70%,-8.32%,-5.33%
Valuetronics Holdings (BN2・電子機器製造受託/無借金),BN2.SI,0.91,-3.19%,-5.21%,-3.70%,-9.90%,-18.46%,+14.68%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Multi-Chem (AWZ・ITセキュリティ/高配当),AWZ.SI,2.50,+0.00%,-2.34%,+2.88%,+9.17%,-28.68%,-20.15%
Yangzijiang Financial Holding (YF8・金融),YF8.SI,0.21,+0.00%,+5.00%,+7.69%,+9.95%,-22.22%,-80.19%
Yangzijiang Shipbuilding (BS6・造船/高収益),BS6.SI,4.69,+1.30%,+1.52%,-1.88%,-7.50%,+14.39%,+43.37%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Riverstone Holdings (AP4・クリーンルーム用手袋),AP4.SI,0.73,-0.68%,-2.03%,-2.03%,-5.84%,+1.25%,-15.11%
Parkway Life REIT (C2PU・医療REIT/安定分配),C2PU.SI,3.98,-0.50%,+0.76%,-1.00%,+0.00%,+2.37%,+2.77%
Sanli Environmental Limited,1E3.SI,0.08,-1.25%,-5.95%,+1.28%,-26.85%,-57.76%,-76.40%


In [7]:
MARKET_TICKERS = {
    # 1. 安定重視・コア保有向け（金融要塞・完全独占）
    "SG Core & Stability (コア保有向け)": {
        "DBS Group (D05.SI・金融要塞/東南アジア最大手/高ROE)": "D05.SI",
        "SGX Group (S68.SI・完全独占/国内唯一の証券取引所)": "S68.SI",
    },
    # 2. 高利回り・インカムゲイン重視（高配当・不動産仲介）
    "SG High Yield (高利回り狙い)": {
        "PropNex (OYY.SI・不動産仲介最大手/高配当・アセットライト)": "OYY.SI",
    },
    # 3. ディープバリュー・ネットキャッシュ
    "SG Value & Margin of Safety (バリュー・資産安全域)": {
        "Valuetronics (BN2.SI・高ネットキャッシュ/高い安全域)": "BN2.SI",
    },
    # 4. 業績復調・半導体関連（高配当＋資本利回り）
    "SG Recovery & Growth (復調・半導体工具)": {
        "Micro-Mechanics (508.SI・無借金/半導体消耗工具/高配当)": "508.SI",
    },
}

# 実行例
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
DBS Group (D05.SI・金融要塞/東南アジア最大手/高ROE),D05.SI,70.88,-1.16%,-3.96%,-4.11%,-9.78%,+23.41%,+39.62%
SGX Group (S68.SI・完全独占/国内唯一の証券取引所),S68.SI,20.24,-0.39%,-0.34%,-0.88%,-2.67%,-2.25%,+22.00%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
PropNex (OYY.SI・不動産仲介最大手/高配当・アセットライト),OYY.SI,1.65,+0.00%,-1.20%,-1.79%,-6.25%,-6.26%,-24.70%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Valuetronics (BN2.SI・高ネットキャッシュ/高い安全域),BN2.SI,0.91,-3.19%,-5.21%,-3.70%,-9.90%,-18.46%,+14.68%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Micro-Mechanics (508.SI・無借金/半導体消耗工具/高配当),508.SI,0.50,+0.00%,+0.00%,+0.00%,-18.70%,-11.50%,-14.50%


In [8]:
MARKET_TICKERS = {
    # 建設・インフラ関連銘柄（高配当・公共需要・改修）
    "SG Construction & Infrastructure (建設・インフラ系)": {
        "BRC Asia (1G6.SI・鉄筋加工最大手/高配当5-5.5%/PER7.5-8.5倍/公共住宅需要)": "BEC.SI",
        "OKP Holdings (5CF.SI・道路土木インフラ/PBR0.8倍割安/無借金経営/PER8-9.5倍)": "5CF.SI",
        "ISOTeam (5WF.SI・HDB建物修繕塗装R&R/小型ターンアラウンド/PER12-14倍)": "5WF.SI",
    },
}

# 実行例
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
BRC Asia (1G6.SI・鉄筋加工最大手/高配当5-5.5%/PER7.5-8.5倍/公共住宅需要),BEC.SI,4.14,+0.24%,+0.98%,+5.34%,+1.94%,-5.16%,+4.86%
OKP Holdings (5CF.SI・道路土木インフラ/PBR0.8倍割安/無借金経営/PER8-9.5倍),5CF.SI,0.63,+1.61%,+3.28%,+4.13%,-7.35%,-18.53%,+15.23%
ISOTeam (5WF.SI・HDB建物修繕塗装R&R/小型ターンアラウンド/PER12-14倍),5WF.SI,0.05,+0.00%,+0.00%,+0.00%,-11.67%,-34.57%,-34.73%


In [9]:
MARKET_TICKERS = {
   "USD ETF": {
        "Vanguard Funds PLC - Vanguard FTSE Global All-Cap UCITS ETF USD Acc (VALL.L)": "VALL.L",
},
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Vanguard Funds PLC - Vanguard FTSE Global All-Cap UCITS ETF USD Acc (VALL.L),VALL.L,3.62,-0.81%,-1.87%,-2.20%,-4.52%,-27.58%,-27.58%


# Gemini Prompt
以下の【分析データ】を基に、指定の【Pythonコードフォーマット】に変換して出力してください。

【出力条件】
・MARKET_TICKERS の辞書形式で出力すること。
・辞書のキーには「企業名 (ティッカー・主要特徴/指標の要約)」を含めること。
・値（Value）には「ティッカーシンボル（.SI付き）」を入れること。
・解説文や導入の挨拶は不要で、そのままコピーして使えるPythonコードのみを出力すること。

【Pythonコードフォーマット例】
MARKET_TICKERS = {
    "カテゴリ名": {
        "銘柄名 (ティッカー・特徴要約)": "ティッカー.SI",
    },
}

【分析データ】
（※ここに分析したテキストや銘柄リスト、あるいは「以下の銘柄を分析してコード化して」と入力）